# Model Training for AI Model

This notebook demonstrates how to train a high-performance AI model. It covers:

1. Loading processed datasets
2. Setting up the model architecture
3. Configuring the training process
4. Training the model
5. Evaluating the model
6. Saving the trained model

## Setup

First, let's install the required packages and import the necessary modules.

In [ ]:
# Install required packages
!pip install transformers datasets torch accelerate bitsandbytes wandb nltk rouge-score pyyaml tqdm

In [ ]:
# Import modules
import os
import sys
import logging
import yaml
import torch
import numpy as np
from datasets import Dataset, load_from_disk
from transformers import AutoTokenizer
from tqdm import tqdm

# Add src directory to path
sys.path.append(os.path.abspath('../'))

# Import custom modules
from src.model.architecture import create_model_from_config
from src.model.training import Trainer, TrainingArguments
from src.utils.metrics import compute_metrics

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Load Configuration

Load the configuration file that defines the model architecture and training parameters.

In [ ]:
# Load configuration
with open("../configs/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Display configuration summary
print(f"Project name: {config['project_name']}")
print(f"Model architecture: {config['model']['architecture']}")
print(f"Model size: {config['model']['size']}")
print(f"Number of training stages: {len(config['training']['stages'])}")

## Load Processed Datasets

Load the processed datasets that were prepared in the previous notebook.

In [ ]:
# Define dataset paths
dataset_dir = "../outputs/processed_datasets"

# Check if datasets exist
if not os.path.exists(dataset_dir):
    print(f"Dataset directory {dataset_dir} not found. Please run the dataset preparation notebook first.")
else:
    # List available datasets
    available_datasets = [f for f in os.listdir(dataset_dir) if f.endswith('.arrow')]
    print(f"Available datasets: {available_datasets}")

    # Load combined dataset
    combined_dataset_path = os.path.join(dataset_dir, "combined_dataset.arrow")
    if os.path.exists(combined_dataset_path):
        combined_dataset = load_from_disk(combined_dataset_path)
        print(f"Loaded combined dataset with {len(combined_dataset)} examples")
    else:
        print(f"Combined dataset not found at {combined_dataset_path}")
        combined_dataset = None
    
    # Load stage datasets
    stage_datasets = {}
    for stage in config['training']['stages']:
        stage_name = stage['name']
        stage_dataset_path = os.path.join(dataset_dir, f"{stage_name}_dataset.arrow")
        
        if os.path.exists(stage_dataset_path):
            stage_datasets[stage_name] = load_from_disk(stage_dataset_path)
            print(f"Loaded {stage_name} dataset with {len(stage_datasets[stage_name])} examples")
        else:
            print(f"{stage_name} dataset not found at {stage_dataset_path}")

## Initialize Tokenizer

Initialize the tokenizer for the model.

In [ ]:
# Initialize tokenizer
tokenizer_name = "gpt2"  # Use an existing tokenizer as a starting point
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Add special tokens if needed
special_tokens = {
    "pad_token": "<pad>",
    "bos_token": "<s>",
    "eos_token": "</s>",
    "unk_token": "<unk>"
}

# Add special tokens to tokenizer
tokenizer.add_special_tokens(special_tokens)

# Update config with tokenizer vocabulary size
config['tokenizer']['vocab_size'] = len(tokenizer)

print(f"Initialized tokenizer with vocabulary size: {len(tokenizer)}")

## Create Model

Create the model architecture based on the configuration.

In [ ]:
# Create model
model = create_model_from_config(config)

# Print model summary
print(f"Created model with architecture: {config['model']['architecture']}")
print(f"Model size: {config['model']['size']}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

## Prepare Data for Training

Prepare the data for training by tokenizing the inputs and targets.

In [ ]:
def tokenize_function(examples):
    """Tokenize inputs and targets for training."""
    # Combine input and target for causal language modeling
    texts = [inp + tokenizer.eos_token + tgt for inp, tgt in zip(examples["input_text"], examples["target_text"])]
    
    # Tokenize texts
    tokenized = tokenizer(texts, padding="max_length", truncation=True, max_length=512, return_tensors="pt")
    
    # Set labels for causal language modeling
    tokenized["labels"] = tokenized["input_ids"].clone()
    
    return tokenized

# Tokenize datasets
tokenized_datasets = {}

for stage_name, dataset in stage_datasets.items():
    # Split dataset into train and validation
    split_dataset = dataset.train_test_split(test_size=0.1)
    
    # Tokenize train and validation sets
    tokenized_train = split_dataset["train"].map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])
    tokenized_val = split_dataset["test"].map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])
    
    # Store tokenized datasets
    tokenized_datasets[stage_name] = {
        "train": tokenized_train,
        "validation": tokenized_val
    }
    
    print(f"Tokenized {stage_name} dataset:")
    print(f"  Train: {len(tokenized_train)} examples")
    print(f"  Validation: {len(tokenized_val)} examples")

## Train Model

Train the model using the prepared datasets and configuration.

In [ ]:
# Train model for each stage
for stage_idx, stage in enumerate(config['training']['stages']):
    stage_name = stage['name']
    
    print(f"\n{'='*50}")
    print(f"Training stage {stage_idx+1}/{len(config['training']['stages'])}: {stage_name}")
    print(f"{'='*50}\n")
    
    # Check if tokenized datasets are available for this stage
    if stage_name not in tokenized_datasets:
        print(f"No tokenized datasets available for {stage_name} stage. Skipping.")
        continue
    
    # Get tokenized datasets for this stage
    train_dataset = tokenized_datasets[stage_name]["train"]
    eval_dataset = tokenized_datasets[stage_name]["validation"]
    
    # Create training arguments
    training_args = TrainingArguments(config, stage_name)
    
    # Create trainer
    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        compute_metrics_fn=lambda preds, labels: compute_metrics(preds, labels, task_type="general")
    )
    
    # Train model
    print(f"Training model for {stage_name} stage...")
    train_results = trainer.train()
    
    # Print training results
    print(f"Training results for {stage_name} stage:")
    for key, value in train_results.items():
        print(f"  {key}: {value}")
    
    # Evaluate model
    print(f"Evaluating model for {stage_name} stage...")
    eval_results = trainer.evaluate()
    
    # Print evaluation results
    print(f"Evaluation results for {stage_name} stage:")
    for key, value in eval_results.items():
        print(f"  {key}: {value}")

## Save Final Model

Save the final trained model for deployment.

In [ ]:
# Create output directory
output_dir = "../outputs/final_model"
os.makedirs(output_dir, exist_ok=True)

# Save model
model.save_pretrained(output_dir)

# Save tokenizer
tokenizer.save_pretrained(output_dir)

# Save configuration
with open(os.path.join(output_dir, "config.yaml"), 'w') as f:
    yaml.dump(config, f)

print(f"Saved final model to {output_dir}")

## Test Model Generation

Test the trained model by generating text from prompts.

In [ ]:
# Define test prompts
test_prompts = [
    "Write a function to calculate the factorial of a number:",
    "Explain the concept of quantum computing to a 10-year-old:",
    "What are the main causes of climate change?"
]

# Generate text from prompts
for prompt in test_prompts:
    # Tokenize prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    # Move to GPU if available
    if torch.cuda.is_available():
        input_ids = input_ids.to("cuda")
        model.to("cuda")
    
    # Generate text
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=200,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode output
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Print results
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated_text}")
    print("-" * 50)

## Summary

In this notebook, we have:

1. Loaded processed datasets
2. Set up the model architecture
3. Configured the training process
4. Trained the model through multiple stages
5. Evaluated the model
6. Saved the trained model
7. Tested the model's generation capabilities

The trained model is now ready for evaluation and deployment.